# ClickBench: DuckDB vs Opteryx on the partitioned `hits` Parquet

Runs the 43 **[ClickBench](https://github.com/ClickHouse/ClickBench)** queries against two embedded SQL engines. Both engines read the **same Parquet files**, the partitioned `hits` dataset that ClickHouse publishes for ClickBench:

| Engine | Reads | Queries and setup taken from |
|---|---|---|
| **DuckDB** | `hits_*.parquet` through a view | the [`duckdb-parquet-partitioned/`](https://github.com/ClickHouse/ClickBench/tree/c9ffb3582d261c9fc83b4140b99de7a81fc025be/duckdb-parquet-partitioned) entry |
| **Opteryx** | `hits_*.parquet` directly, no view | the [`opteryx/`](https://github.com/ClickHouse/ClickBench/tree/c9ffb3582d261c9fc83b4140b99de7a81fc025be/opteryx) entry |

The pipeline is: download the first `NUM_FILES` of the 100 `hits` partitions, run all 43 queries on both engines, cross-check row counts, and compare wall-clock time.

This notebook accompanies [Benchmarking](https://docs.opteryx.app/docs/reference/internals/benchmarking) on docs.opteryx.app. The numbers it produces belong to whatever machine runs it.

> **These results cannot be compared with the official ClickBench results.** The [published ClickBench results](https://benchmark.clickhouse.com/) run all 100 partitions (about 100M rows, 14 GB of Parquet) on a 16-vCPU `c6a.4xlarge`, and count a cold first run. This notebook queries a subset of the partitions on a small, shared Colab runtime and reports the best of `ITERATIONS` warm runs. Several queries filter on one counter and one month, or skip the first 1,000 or 10,000 groups with `OFFSET`, so on a subset they return few or no rows and do much less work than on the full dataset. Compare the two engines with each other, on this run, and nothing else.

**Runtime note:** each partition is about 1M rows and 120 to 230 MB. At the default `NUM_FILES = 10` (about 10M rows, 1.5 GB to download) the whole run takes several minutes on a free Colab CPU runtime. For a quick smoke test, set `NUM_FILES = 1` and `ITERATIONS = 1`.

In [ ]:
# --- Configuration -----------------------------------------------------
NUM_FILES = 10           # how many of the 100 hits_N.parquet partitions to query
                         # (~1M rows each). At 10 the larger engine process
                         # peaks at about 6 GB, which fits a free Colab runtime;
                         # 20 peaks near 9 GB. The full ClickBench dataset is 100.
ITERATIONS = 3           # warm runs per query per engine; the minimum is reported
import os
WORK_DIR = os.path.abspath("/content/clickbench" if os.path.isdir("/content") else "./clickbench")

os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print("Working directory:", os.getcwd())

cpus = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
print("CPUs available:", cpus)

## 0. Install engines

`duckdb` and `opteryx-core` install from PyPI. Both engines run with their default settings, as they do in the ClickBench entries.

In [ ]:
%pip install -q --upgrade duckdb opteryx-core pandas matplotlib

import duckdb
import opteryx

print("duckdb  ", duckdb.__version__)
print("opteryx ", opteryx.__version__)

## 1. Download the `hits` partitions

ClickHouse publishes the ClickBench dataset as 100 Parquet files, `hits_0.parquet` to `hits_99.parquet`, at `datasets.clickhouse.com`. These are the files the ClickBench `duckdb-parquet-partitioned` and `opteryx` entries download.

Files go into `downloads/` and are skipped if they are already there, so re-running this cell, or raising `NUM_FILES` later, only fetches what is missing. The first `NUM_FILES` of them are then linked into `hits/`, the folder both engines query, so lowering `NUM_FILES` takes effect without deleting anything.

In [ ]:
import shutil
import time
import urllib.request

BASE_URL = "https://datasets.clickhouse.com/hits_compatible/athena_partitioned"
DOWNLOAD_DIR = os.path.join(WORK_DIR, "downloads")
HITS_DIR = os.path.join(WORK_DIR, "hits")
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

assert 1 <= NUM_FILES <= 100, "NUM_FILES must be between 1 and 100"
files = [f"hits_{i}.parquet" for i in range(NUM_FILES)]

for n, name in enumerate(files, start=1):
    path = os.path.join(DOWNLOAD_DIR, name)
    if os.path.exists(path):
        print(f"[{n}/{NUM_FILES}] {name}  {os.path.getsize(path) / 1e6:7.1f} MB  already downloaded")
        continue
    t0 = time.perf_counter()
    partial = path + ".part"
    # The bucket refuses urllib's default User-Agent with a 403, so send our own.
    request = urllib.request.Request(f"{BASE_URL}/{name}", headers={"User-Agent": "clickbench-notebook"})
    with urllib.request.urlopen(request) as response, open(partial, "wb") as fh:
        shutil.copyfileobj(response, fh, length=4 * 1024 * 1024)
    os.replace(partial, path)   # only complete files get the final name
    elapsed = time.perf_counter() - t0
    size_mb = os.path.getsize(path) / 1e6
    print(f"[{n}/{NUM_FILES}] {name}  {size_mb:7.1f} MB  in {elapsed:5.1f}s ({size_mb / elapsed:.0f} MB/s)")

# Rebuild hits/ with exactly the first NUM_FILES partitions.
shutil.rmtree(HITS_DIR, ignore_errors=True)
os.makedirs(HITS_DIR)
for name in files:
    src, dst = os.path.join(DOWNLOAD_DIR, name), os.path.join(HITS_DIR, name)
    try:
        os.link(src, dst)
    except OSError:
        shutil.copyfile(src, dst)

total_mb = sum(os.path.getsize(os.path.join(HITS_DIR, f)) for f in files) / 1e6
print(f"\n{NUM_FILES} partition(s), {total_mb:,.0f} MB of Parquet in {HITS_DIR}")

## 2. Both engines read the Parquet directly

The [TPC-H notebook](https://docs.opteryx.app/notebooks/tpch-duckdb-vs-opteryx.ipynb) converts Parquet to `.skene`, Opteryx's native file format, before querying. This notebook does not. It follows the official [`opteryx/` ClickBench entry](https://github.com/ClickHouse/ClickBench/tree/c9ffb3582d261c9fc83b4140b99de7a81fc025be/opteryx), which queries the partitioned Parquet files as they are downloaded, with no conversion, preloading or view. That entry's `load` step only moves the files into a `hits/` folder, and Opteryx resolves `FROM hits` to every Parquet file in `./hits/`. Querying data where it already lives is the case Opteryx is built for, so that is the case ClickBench measures.

DuckDB also reads the Parquet, through the view that the [`duckdb-parquet-partitioned/`](https://github.com/ClickHouse/ClickBench/tree/c9ffb3582d261c9fc83b4140b99de7a81fc025be/duckdb-parquet-partitioned) entry creates in its `create.sql`. The view exposes the files as strings rather than binary (`binary_as_string=True`) and turns the integer `EventDate` into a `DATE`. A `toDateTime` macro converts the integer `EventTime`. The only change here is the file glob, which points into `hits/`. The connection also sets `parquet_metadata_cache`, as that entry's `query` script does. The benchmark script in §4 sets all of this up.

Neither engine gets an import step or a native-format copy, so both start from the same bytes on disk.

## 3. The 43 ClickBench queries

Each engine runs the `queries.sql` from its own ClickBench entry, copied unchanged from [ClickHouse/ClickBench@c9ffb35](https://github.com/ClickHouse/ClickBench/tree/c9ffb3582d261c9fc83b4140b99de7a81fc025be). The two files are the standard ClickBench queries. They differ only where an engine's dialect needs it, and the [`opteryx/` README](https://github.com/ClickHouse/ClickBench/tree/c9ffb3582d261c9fc83b4140b99de7a81fc025be/opteryx#query-dialect) lists every Opteryx adaptation:

| Queries | Opteryx | DuckDB |
|---|---|---|
| Q19, Q43 | `EventTime::TIMESTAMP[s]` | `toDateTime(EventTime)` macro |
| Q43 | `TRUNC(ts, 'minute')` | `DATE_TRUNC('minute', ts)` |
| Q28, Q29 | `length(...)` | `STRLEN(...)` |
| Q29 | `b''` and `r''` regex literals | plain string literals |
| Q37 to Q43 | `EventDate::DATE >= '2013-07-01'::DATE` | `EventDate >= '2013-07-01'` (the view already made it a `DATE`) |

Queries are numbered from Q01, in file order. They are saved to `queries.json` for the benchmark script in the next step.

In [ ]:
import json

# opteryx/queries.sql
OPTERYX_SQL = r"""
SELECT COUNT(*) FROM hits;
SELECT COUNT(*) FROM hits WHERE AdvEngineID <> 0;
SELECT SUM(AdvEngineID), COUNT(*), AVG(ResolutionWidth) FROM hits;
SELECT AVG(UserID) FROM hits;
SELECT COUNT(DISTINCT UserID) FROM hits;
SELECT COUNT(DISTINCT SearchPhrase) FROM hits;
SELECT MIN(EventDate), MAX(EventDate) FROM hits;
SELECT AdvEngineID, COUNT(*) FROM hits WHERE AdvEngineID <> 0 GROUP BY AdvEngineID ORDER BY COUNT(*) DESC;
SELECT RegionID, COUNT(DISTINCT UserID) AS u FROM hits GROUP BY RegionID ORDER BY u DESC LIMIT 10;
SELECT RegionID, SUM(AdvEngineID), COUNT(*) AS c, AVG(ResolutionWidth), COUNT(DISTINCT UserID) FROM hits GROUP BY RegionID ORDER BY c DESC LIMIT 10;
SELECT MobilePhoneModel, COUNT(DISTINCT UserID) AS u FROM hits WHERE MobilePhoneModel <> '' GROUP BY MobilePhoneModel ORDER BY u DESC LIMIT 10;
SELECT MobilePhone, MobilePhoneModel, COUNT(DISTINCT UserID) AS u FROM hits WHERE MobilePhoneModel <> '' GROUP BY MobilePhone, MobilePhoneModel ORDER BY u DESC LIMIT 10;
SELECT SearchPhrase, COUNT(*) AS c FROM hits WHERE SearchPhrase <> '' GROUP BY SearchPhrase ORDER BY c DESC LIMIT 10;
SELECT SearchPhrase, COUNT(DISTINCT UserID) AS u FROM hits WHERE SearchPhrase <> '' GROUP BY SearchPhrase ORDER BY u DESC LIMIT 10;
SELECT SearchEngineID, SearchPhrase, COUNT(*) AS c FROM hits WHERE SearchPhrase <> '' GROUP BY SearchEngineID, SearchPhrase ORDER BY c DESC LIMIT 10;
SELECT UserID, COUNT(*) FROM hits GROUP BY UserID ORDER BY COUNT(*) DESC LIMIT 10;
SELECT UserID, SearchPhrase, COUNT(*) FROM hits GROUP BY UserID, SearchPhrase ORDER BY COUNT(*) DESC LIMIT 10;
SELECT UserID, SearchPhrase, COUNT(*) FROM hits GROUP BY UserID, SearchPhrase LIMIT 10;
SELECT UserID, extract(minute FROM EventTime::TIMESTAMP[s]) AS m, SearchPhrase, COUNT(*) FROM hits GROUP BY UserID, extract(minute FROM EventTime::TIMESTAMP[s]), SearchPhrase ORDER BY COUNT(*) DESC LIMIT 10;
SELECT UserID FROM hits WHERE UserID = 435090932899640449;
SELECT COUNT(*) FROM hits WHERE URL LIKE '%google%';
SELECT SearchPhrase, MIN(URL), COUNT(*) AS c FROM hits WHERE URL LIKE '%google%' AND SearchPhrase <> '' GROUP BY SearchPhrase ORDER BY c DESC LIMIT 10;
SELECT SearchPhrase, MIN(URL), MIN(Title), COUNT(*) AS c, COUNT(DISTINCT UserID) FROM hits WHERE Title LIKE '%Google%' AND URL NOT LIKE '%.google.%' AND SearchPhrase <> '' GROUP BY SearchPhrase ORDER BY c DESC LIMIT 10;
SELECT * FROM hits WHERE URL LIKE '%google%' ORDER BY EventTime LIMIT 10;
SELECT SearchPhrase FROM hits WHERE SearchPhrase <> '' ORDER BY EventTime LIMIT 10;
SELECT SearchPhrase FROM hits WHERE SearchPhrase <> '' ORDER BY SearchPhrase LIMIT 10;
SELECT SearchPhrase FROM hits WHERE SearchPhrase <> '' ORDER BY EventTime, SearchPhrase LIMIT 10;
SELECT CounterID, AVG(length(URL)) AS l, COUNT(*) AS c FROM hits WHERE URL <> '' GROUP BY CounterID HAVING COUNT(*) > 100000 ORDER BY l DESC LIMIT 25;
SELECT REGEXP_REPLACE(Referer, b'^https?://(?:www\.)?([^/]+)/.*$', r'\1') AS k, AVG(length(Referer)) AS l, COUNT(*) AS c, MIN(Referer) FROM hits WHERE Referer <> '' GROUP BY REGEXP_REPLACE(Referer, b'^https?://(?:www\.)?([^/]+)/.*$', r'\1') HAVING COUNT(*) > 100000 ORDER BY l DESC LIMIT 25;
SELECT SUM(ResolutionWidth), SUM(ResolutionWidth + 1), SUM(ResolutionWidth + 2), SUM(ResolutionWidth + 3), SUM(ResolutionWidth + 4), SUM(ResolutionWidth + 5), SUM(ResolutionWidth + 6), SUM(ResolutionWidth + 7), SUM(ResolutionWidth + 8), SUM(ResolutionWidth + 9), SUM(ResolutionWidth + 10), SUM(ResolutionWidth + 11), SUM(ResolutionWidth + 12), SUM(ResolutionWidth + 13), SUM(ResolutionWidth + 14), SUM(ResolutionWidth + 15), SUM(ResolutionWidth + 16), SUM(ResolutionWidth + 17), SUM(ResolutionWidth + 18), SUM(ResolutionWidth + 19), SUM(ResolutionWidth + 20), SUM(ResolutionWidth + 21), SUM(ResolutionWidth + 22), SUM(ResolutionWidth + 23), SUM(ResolutionWidth + 24), SUM(ResolutionWidth + 25), SUM(ResolutionWidth + 26), SUM(ResolutionWidth + 27), SUM(ResolutionWidth + 28), SUM(ResolutionWidth + 29), SUM(ResolutionWidth + 30), SUM(ResolutionWidth + 31), SUM(ResolutionWidth + 32), SUM(ResolutionWidth + 33), SUM(ResolutionWidth + 34), SUM(ResolutionWidth + 35), SUM(ResolutionWidth + 36), SUM(ResolutionWidth + 37), SUM(ResolutionWidth + 38), SUM(ResolutionWidth + 39), SUM(ResolutionWidth + 40), SUM(ResolutionWidth + 41), SUM(ResolutionWidth + 42), SUM(ResolutionWidth + 43), SUM(ResolutionWidth + 44), SUM(ResolutionWidth + 45), SUM(ResolutionWidth + 46), SUM(ResolutionWidth + 47), SUM(ResolutionWidth + 48), SUM(ResolutionWidth + 49), SUM(ResolutionWidth + 50), SUM(ResolutionWidth + 51), SUM(ResolutionWidth + 52), SUM(ResolutionWidth + 53), SUM(ResolutionWidth + 54), SUM(ResolutionWidth + 55), SUM(ResolutionWidth + 56), SUM(ResolutionWidth + 57), SUM(ResolutionWidth + 58), SUM(ResolutionWidth + 59), SUM(ResolutionWidth + 60), SUM(ResolutionWidth + 61), SUM(ResolutionWidth + 62), SUM(ResolutionWidth + 63), SUM(ResolutionWidth + 64), SUM(ResolutionWidth + 65), SUM(ResolutionWidth + 66), SUM(ResolutionWidth + 67), SUM(ResolutionWidth + 68), SUM(ResolutionWidth + 69), SUM(ResolutionWidth + 70), SUM(ResolutionWidth + 71), SUM(ResolutionWidth + 72), SUM(ResolutionWidth + 73), SUM(ResolutionWidth + 74), SUM(ResolutionWidth + 75), SUM(ResolutionWidth + 76), SUM(ResolutionWidth + 77), SUM(ResolutionWidth + 78), SUM(ResolutionWidth + 79), SUM(ResolutionWidth + 80), SUM(ResolutionWidth + 81), SUM(ResolutionWidth + 82), SUM(ResolutionWidth + 83), SUM(ResolutionWidth + 84), SUM(ResolutionWidth + 85), SUM(ResolutionWidth + 86), SUM(ResolutionWidth + 87), SUM(ResolutionWidth + 88), SUM(ResolutionWidth + 89) FROM hits;
SELECT SearchEngineID, ClientIP, COUNT(*) AS c, SUM(IsRefresh), AVG(ResolutionWidth) FROM hits WHERE SearchPhrase <> '' GROUP BY SearchEngineID, ClientIP ORDER BY c DESC LIMIT 10;
SELECT WatchID, ClientIP, COUNT(*) AS c, SUM(IsRefresh), AVG(ResolutionWidth) FROM hits WHERE SearchPhrase <> '' GROUP BY WatchID, ClientIP ORDER BY c DESC LIMIT 10;
SELECT WatchID, ClientIP, COUNT(*) AS c, SUM(IsRefresh), AVG(ResolutionWidth) FROM hits GROUP BY WatchID, ClientIP ORDER BY c DESC LIMIT 10;
SELECT URL, COUNT(*) AS c FROM hits GROUP BY URL ORDER BY c DESC LIMIT 10;
SELECT 1, URL, COUNT(*) AS c FROM hits GROUP BY 1, URL ORDER BY c DESC LIMIT 10;
SELECT ClientIP, ClientIP - 1, ClientIP - 2, ClientIP - 3, COUNT(*) AS c FROM hits GROUP BY ClientIP, ClientIP - 1, ClientIP - 2, ClientIP - 3 ORDER BY c DESC LIMIT 10;
SELECT URL, COUNT(*) AS PageViews FROM hits WHERE CounterID = 62 AND EventDate::DATE >= '2013-07-01'::DATE AND EventDate::DATE <= '2013-07-31'::DATE AND DontCountHits = 0 AND IsRefresh = 0 AND URL <> '' GROUP BY URL ORDER BY PageViews DESC LIMIT 10;
SELECT Title, COUNT(*) AS PageViews FROM hits WHERE CounterID = 62 AND EventDate::DATE >= '2013-07-01'::DATE AND EventDate::DATE <= '2013-07-31'::DATE AND DontCountHits = 0 AND IsRefresh = 0 AND Title <> '' GROUP BY Title ORDER BY PageViews DESC LIMIT 10;
SELECT URL, COUNT(*) AS PageViews FROM hits WHERE CounterID = 62 AND EventDate::DATE >= '2013-07-01'::DATE AND EventDate::DATE <= '2013-07-31'::DATE AND IsRefresh = 0 AND IsLink <> 0 AND IsDownload = 0 GROUP BY URL ORDER BY PageViews DESC LIMIT 10 OFFSET 1000;
SELECT TraficSourceID, SearchEngineID, AdvEngineID, CASE WHEN (SearchEngineID = 0 AND AdvEngineID = 0) THEN Referer ELSE '' END AS Src, URL AS Dst, COUNT(*) AS PageViews FROM hits WHERE CounterID = 62 AND EventDate::DATE >= '2013-07-01'::DATE AND EventDate::DATE <= '2013-07-31'::DATE AND IsRefresh = 0 GROUP BY TraficSourceID, SearchEngineID, AdvEngineID, CASE WHEN (SearchEngineID = 0 AND AdvEngineID = 0) THEN Referer ELSE '' END, URL ORDER BY PageViews DESC LIMIT 10 OFFSET 1000;
SELECT URLHash, EventDate, COUNT(*) AS PageViews FROM hits WHERE CounterID = 62 AND EventDate::DATE >= '2013-07-01'::DATE AND EventDate::DATE <= '2013-07-31'::DATE AND IsRefresh = 0 AND TraficSourceID IN (-1, 6) AND RefererHash = 3594120000172545465 GROUP BY URLHash, EventDate ORDER BY PageViews DESC LIMIT 10 OFFSET 100;
SELECT WindowClientWidth, WindowClientHeight, COUNT(*) AS PageViews FROM hits WHERE CounterID = 62 AND EventDate::DATE >= '2013-07-01'::DATE AND EventDate::DATE <= '2013-07-31'::DATE AND IsRefresh = 0 AND DontCountHits = 0 AND URLHash = 2868770270353813622 GROUP BY WindowClientWidth, WindowClientHeight ORDER BY PageViews DESC LIMIT 10 OFFSET 10000;
SELECT TRUNC(EventTime::TIMESTAMP[s], 'minute') AS M, COUNT(*) AS PageViews FROM hits WHERE CounterID = 62 AND EventDate::DATE >= '2013-07-14'::DATE AND EventDate::DATE <= '2013-07-15'::DATE AND IsRefresh = 0 AND DontCountHits = 0 GROUP BY TRUNC(EventTime::TIMESTAMP[s], 'minute') ORDER BY M LIMIT 10 OFFSET 1000;
"""

# duckdb-parquet-partitioned/queries.sql
DUCKDB_SQL = r"""
SELECT COUNT(*) FROM hits;
SELECT COUNT(*) FROM hits WHERE AdvEngineID <> 0;
SELECT SUM(AdvEngineID), COUNT(*), AVG(ResolutionWidth) FROM hits;
SELECT AVG(UserID) FROM hits;
SELECT COUNT(DISTINCT UserID) FROM hits;
SELECT COUNT(DISTINCT SearchPhrase) FROM hits;
SELECT MIN(EventDate), MAX(EventDate) FROM hits;
SELECT AdvEngineID, COUNT(*) FROM hits WHERE AdvEngineID <> 0 GROUP BY AdvEngineID ORDER BY COUNT(*) DESC;
SELECT RegionID, COUNT(DISTINCT UserID) AS u FROM hits GROUP BY RegionID ORDER BY u DESC LIMIT 10;
SELECT RegionID, SUM(AdvEngineID), COUNT(*) AS c, AVG(ResolutionWidth), COUNT(DISTINCT UserID) FROM hits GROUP BY RegionID ORDER BY c DESC LIMIT 10;
SELECT MobilePhoneModel, COUNT(DISTINCT UserID) AS u FROM hits WHERE MobilePhoneModel <> '' GROUP BY MobilePhoneModel ORDER BY u DESC LIMIT 10;
SELECT MobilePhone, MobilePhoneModel, COUNT(DISTINCT UserID) AS u FROM hits WHERE MobilePhoneModel <> '' GROUP BY MobilePhone, MobilePhoneModel ORDER BY u DESC LIMIT 10;
SELECT SearchPhrase, COUNT(*) AS c FROM hits WHERE SearchPhrase <> '' GROUP BY SearchPhrase ORDER BY c DESC LIMIT 10;
SELECT SearchPhrase, COUNT(DISTINCT UserID) AS u FROM hits WHERE SearchPhrase <> '' GROUP BY SearchPhrase ORDER BY u DESC LIMIT 10;
SELECT SearchEngineID, SearchPhrase, COUNT(*) AS c FROM hits WHERE SearchPhrase <> '' GROUP BY SearchEngineID, SearchPhrase ORDER BY c DESC LIMIT 10;
SELECT UserID, COUNT(*) FROM hits GROUP BY UserID ORDER BY COUNT(*) DESC LIMIT 10;
SELECT UserID, SearchPhrase, COUNT(*) FROM hits GROUP BY UserID, SearchPhrase ORDER BY COUNT(*) DESC LIMIT 10;
SELECT UserID, SearchPhrase, COUNT(*) FROM hits GROUP BY UserID, SearchPhrase LIMIT 10;
SELECT UserID, extract(minute FROM toDateTime(EventTime)) AS m, SearchPhrase, COUNT(*) FROM hits GROUP BY UserID, m, SearchPhrase ORDER BY COUNT(*) DESC LIMIT 10;
SELECT UserID FROM hits WHERE UserID = 435090932899640449;
SELECT COUNT(*) FROM hits WHERE URL LIKE '%google%';
SELECT SearchPhrase, MIN(URL), COUNT(*) AS c FROM hits WHERE URL LIKE '%google%' AND SearchPhrase <> '' GROUP BY SearchPhrase ORDER BY c DESC LIMIT 10;
SELECT SearchPhrase, MIN(URL), MIN(Title), COUNT(*) AS c, COUNT(DISTINCT UserID) FROM hits WHERE Title LIKE '%Google%' AND URL NOT LIKE '%.google.%' AND SearchPhrase <> '' GROUP BY SearchPhrase ORDER BY c DESC LIMIT 10;
SELECT * FROM hits WHERE URL LIKE '%google%' ORDER BY EventTime LIMIT 10;
SELECT SearchPhrase FROM hits WHERE SearchPhrase <> '' ORDER BY EventTime LIMIT 10;
SELECT SearchPhrase FROM hits WHERE SearchPhrase <> '' ORDER BY SearchPhrase LIMIT 10;
SELECT SearchPhrase FROM hits WHERE SearchPhrase <> '' ORDER BY EventTime, SearchPhrase LIMIT 10;
SELECT CounterID, AVG(STRLEN(URL)) AS l, COUNT(*) AS c FROM hits WHERE URL <> '' GROUP BY CounterID HAVING COUNT(*) > 100000 ORDER BY l DESC LIMIT 25;
SELECT REGEXP_REPLACE(Referer, '^https?://(?:www\.)?([^/]+)/.*$', '\1') AS k, AVG(STRLEN(Referer)) AS l, COUNT(*) AS c, MIN(Referer) FROM hits WHERE Referer <> '' GROUP BY k HAVING COUNT(*) > 100000 ORDER BY l DESC LIMIT 25;
SELECT SUM(ResolutionWidth), SUM(ResolutionWidth + 1), SUM(ResolutionWidth + 2), SUM(ResolutionWidth + 3), SUM(ResolutionWidth + 4), SUM(ResolutionWidth + 5), SUM(ResolutionWidth + 6), SUM(ResolutionWidth + 7), SUM(ResolutionWidth + 8), SUM(ResolutionWidth + 9), SUM(ResolutionWidth + 10), SUM(ResolutionWidth + 11), SUM(ResolutionWidth + 12), SUM(ResolutionWidth + 13), SUM(ResolutionWidth + 14), SUM(ResolutionWidth + 15), SUM(ResolutionWidth + 16), SUM(ResolutionWidth + 17), SUM(ResolutionWidth + 18), SUM(ResolutionWidth + 19), SUM(ResolutionWidth + 20), SUM(ResolutionWidth + 21), SUM(ResolutionWidth + 22), SUM(ResolutionWidth + 23), SUM(ResolutionWidth + 24), SUM(ResolutionWidth + 25), SUM(ResolutionWidth + 26), SUM(ResolutionWidth + 27), SUM(ResolutionWidth + 28), SUM(ResolutionWidth + 29), SUM(ResolutionWidth + 30), SUM(ResolutionWidth + 31), SUM(ResolutionWidth + 32), SUM(ResolutionWidth + 33), SUM(ResolutionWidth + 34), SUM(ResolutionWidth + 35), SUM(ResolutionWidth + 36), SUM(ResolutionWidth + 37), SUM(ResolutionWidth + 38), SUM(ResolutionWidth + 39), SUM(ResolutionWidth + 40), SUM(ResolutionWidth + 41), SUM(ResolutionWidth + 42), SUM(ResolutionWidth + 43), SUM(ResolutionWidth + 44), SUM(ResolutionWidth + 45), SUM(ResolutionWidth + 46), SUM(ResolutionWidth + 47), SUM(ResolutionWidth + 48), SUM(ResolutionWidth + 49), SUM(ResolutionWidth + 50), SUM(ResolutionWidth + 51), SUM(ResolutionWidth + 52), SUM(ResolutionWidth + 53), SUM(ResolutionWidth + 54), SUM(ResolutionWidth + 55), SUM(ResolutionWidth + 56), SUM(ResolutionWidth + 57), SUM(ResolutionWidth + 58), SUM(ResolutionWidth + 59), SUM(ResolutionWidth + 60), SUM(ResolutionWidth + 61), SUM(ResolutionWidth + 62), SUM(ResolutionWidth + 63), SUM(ResolutionWidth + 64), SUM(ResolutionWidth + 65), SUM(ResolutionWidth + 66), SUM(ResolutionWidth + 67), SUM(ResolutionWidth + 68), SUM(ResolutionWidth + 69), SUM(ResolutionWidth + 70), SUM(ResolutionWidth + 71), SUM(ResolutionWidth + 72), SUM(ResolutionWidth + 73), SUM(ResolutionWidth + 74), SUM(ResolutionWidth + 75), SUM(ResolutionWidth + 76), SUM(ResolutionWidth + 77), SUM(ResolutionWidth + 78), SUM(ResolutionWidth + 79), SUM(ResolutionWidth + 80), SUM(ResolutionWidth + 81), SUM(ResolutionWidth + 82), SUM(ResolutionWidth + 83), SUM(ResolutionWidth + 84), SUM(ResolutionWidth + 85), SUM(ResolutionWidth + 86), SUM(ResolutionWidth + 87), SUM(ResolutionWidth + 88), SUM(ResolutionWidth + 89) FROM hits;
SELECT SearchEngineID, ClientIP, COUNT(*) AS c, SUM(IsRefresh), AVG(ResolutionWidth) FROM hits WHERE SearchPhrase <> '' GROUP BY SearchEngineID, ClientIP ORDER BY c DESC LIMIT 10;
SELECT WatchID, ClientIP, COUNT(*) AS c, SUM(IsRefresh), AVG(ResolutionWidth) FROM hits WHERE SearchPhrase <> '' GROUP BY WatchID, ClientIP ORDER BY c DESC LIMIT 10;
SELECT WatchID, ClientIP, COUNT(*) AS c, SUM(IsRefresh), AVG(ResolutionWidth) FROM hits GROUP BY WatchID, ClientIP ORDER BY c DESC LIMIT 10;
SELECT URL, COUNT(*) AS c FROM hits GROUP BY URL ORDER BY c DESC LIMIT 10;
SELECT 1, URL, COUNT(*) AS c FROM hits GROUP BY 1, URL ORDER BY c DESC LIMIT 10;
SELECT ClientIP, ClientIP - 1, ClientIP - 2, ClientIP - 3, COUNT(*) AS c FROM hits GROUP BY ClientIP, ClientIP - 1, ClientIP - 2, ClientIP - 3 ORDER BY c DESC LIMIT 10;
SELECT URL, COUNT(*) AS PageViews FROM hits WHERE CounterID = 62 AND EventDate >= '2013-07-01' AND EventDate <= '2013-07-31' AND DontCountHits = 0 AND IsRefresh = 0 AND URL <> '' GROUP BY URL ORDER BY PageViews DESC LIMIT 10;
SELECT Title, COUNT(*) AS PageViews FROM hits WHERE CounterID = 62 AND EventDate >= '2013-07-01' AND EventDate <= '2013-07-31' AND DontCountHits = 0 AND IsRefresh = 0 AND Title <> '' GROUP BY Title ORDER BY PageViews DESC LIMIT 10;
SELECT URL, COUNT(*) AS PageViews FROM hits WHERE CounterID = 62 AND EventDate >= '2013-07-01' AND EventDate <= '2013-07-31' AND IsRefresh = 0 AND IsLink <> 0 AND IsDownload = 0 GROUP BY URL ORDER BY PageViews DESC LIMIT 10 OFFSET 1000;
SELECT TraficSourceID, SearchEngineID, AdvEngineID, CASE WHEN (SearchEngineID = 0 AND AdvEngineID = 0) THEN Referer ELSE '' END AS Src, URL AS Dst, COUNT(*) AS PageViews FROM hits WHERE CounterID = 62 AND EventDate >= '2013-07-01' AND EventDate <= '2013-07-31' AND IsRefresh = 0 GROUP BY TraficSourceID, SearchEngineID, AdvEngineID, Src, Dst ORDER BY PageViews DESC LIMIT 10 OFFSET 1000;
SELECT URLHash, EventDate, COUNT(*) AS PageViews FROM hits WHERE CounterID = 62 AND EventDate >= '2013-07-01' AND EventDate <= '2013-07-31' AND IsRefresh = 0 AND TraficSourceID IN (-1, 6) AND RefererHash = 3594120000172545465 GROUP BY URLHash, EventDate ORDER BY PageViews DESC LIMIT 10 OFFSET 100;
SELECT WindowClientWidth, WindowClientHeight, COUNT(*) AS PageViews FROM hits WHERE CounterID = 62 AND EventDate >= '2013-07-01' AND EventDate <= '2013-07-31' AND IsRefresh = 0 AND DontCountHits = 0 AND URLHash = 2868770270353813622 GROUP BY WindowClientWidth, WindowClientHeight ORDER BY PageViews DESC LIMIT 10 OFFSET 10000;
SELECT DATE_TRUNC('minute', toDateTime(EventTime)) AS M, COUNT(*) AS PageViews FROM hits WHERE CounterID = 62 AND EventDate >= '2013-07-14' AND EventDate <= '2013-07-15' AND IsRefresh = 0 AND DontCountHits = 0 GROUP BY DATE_TRUNC('minute', toDateTime(EventTime)) ORDER BY DATE_TRUNC('minute', toDateTime(EventTime)) LIMIT 10 OFFSET 1000;
"""

QUERIES = {
    f"Q{i:02d}": {"opteryx": opteryx_q, "duckdb": duckdb_q}
    for i, (opteryx_q, duckdb_q) in enumerate(
        zip(OPTERYX_SQL.strip().splitlines(), DUCKDB_SQL.strip().splitlines(), strict=True), start=1
    )
}

with open("queries.json", "w") as fh:
    json.dump(QUERIES, fh)
print(f"{len(QUERIES)} queries saved to queries.json: {list(QUERIES)}")

## 4. Benchmark runner

The benchmark is a script, and each engine runs it in **its own fresh process**, the way the [Opteryx performance suites](https://docs.opteryx.app/docs/reference/internals/benchmarking#method) do. Running it in a separate process is what makes the allocator choice in step 5 possible. It also means neither engine inherits anything from the other, or from this notebook's kernel.

Inside the script:

- **Warm-up first.** The first query in a fresh process pays one-off costs: loading native libraries, starting thread pools, and the first read of each file from disk. On a new runtime that can take seconds. Each engine runs a `COUNT(*)` over `hits` first and reports it separately as the cold start, along with the row count it saw, so the cost isn't charged to whichever query happens to run first.
- **DuckDB** gets one connection, reused across all queries, which is normal embedded-engine usage.
- **Opteryx** opens a fresh session per iteration and drains the result as morsels without rendering it. That is the span the [`opteryx/` ClickBench entry](https://github.com/ClickHouse/ClickBench/blob/c9ffb3582d261c9fc83b4140b99de7a81fc025be/opteryx/server.py) times. It leaves its worker pools at their defaults, which it sizes to the machine, as the official entry does.
- Each query runs `ITERATIONS` warm iterations with a garbage collection before each, and the **minimum** time is reported.
- The script records its own **peak memory**, so §6 can show what each engine needed.

Row counts are recorded from both engines as a correctness check. Both engines query the same files, so they should return the same number of rows for every query.

In [ ]:
%%writefile bench_clickbench.py
"""Run the ClickBench queries in queries.json on one engine and write the results as JSON.

Usage: python bench_clickbench.py {duckdb|opteryx} <iterations> <output.json>
"""
import gc
import json
import resource
import statistics
import sys
import time

engine, iterations, out_path = sys.argv[1], int(sys.argv[2]), sys.argv[3]
with open("queries.json") as fh:
    QUERIES = json.load(fh)

if engine == "duckdb":
    import duckdb

    # The view and macro from the duckdb-parquet-partitioned entry's create.sql,
    # with the glob pointed into hits/, and the setting from its query script.
    con = duckdb.connect()
    con.execute("SET parquet_metadata_cache=true")
    con.execute("""
        CREATE VIEW hits AS
        SELECT *
            REPLACE (make_date(EventDate) AS EventDate)
        FROM read_parquet('hits/hits_*.parquet', binary_as_string=True);
    """)
    con.execute("CREATE MACRO toDateTime(t) AS epoch_ms(t * 1000);")

    def run(sql):
        t0 = time.perf_counter()
        result = con.execute(sql).fetchall()
        return (time.perf_counter() - t0) * 1000, len(result), result

elif engine == "opteryx":
    import opteryx

    # No setup: `FROM hits` resolves to the Parquet files in ./hits/.
    def run(sql):
        session = opteryx.session()
        try:
            rows, first = 0, None
            t0 = time.perf_counter()
            for morsel in session.execute_to_morsels(sql):
                if morsel is not None and hasattr(morsel, "num_rows"):
                    if first is None and morsel.num_rows:
                        first = morsel
                    rows += morsel.num_rows
            elapsed = (time.perf_counter() - t0) * 1000
            return elapsed, rows, [first[0]] if first is not None else []
        finally:
            session.close()

else:
    raise SystemExit(f"unknown engine: {engine}")

elapsed, _, result = run("SELECT COUNT(*) FROM hits")
hits_rows = result[0][0]
print(f"{engine} cold start: {elapsed:,.1f}ms (COUNT(*) FROM hits = {hits_rows:,} rows)", flush=True)

results = {}
suite_start = time.perf_counter()
for name, sql_by_engine in QUERIES.items():
    times = []
    try:
        for _ in range(iterations):
            gc.collect()
            elapsed_ms, rows, _ = run(sql_by_engine[engine])
            times.append(elapsed_ms)
        results[name] = {
            "ms": min(times),
            "median_ms": statistics.median(times),
            "rows": rows,
            "error": None,
        }
        print(f"  {name}  {min(times):>10,.1f}ms", flush=True)
    except Exception as exc:
        results[name] = {"ms": None, "median_ms": None, "rows": None,
                         "error": f"{type(exc).__name__}: {exc}"}
        print(f"  {name}  FAILED  {results[name]['error']}", flush=True)

# ru_maxrss is in KiB on Linux and bytes on macOS.
peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
peak_gb = peak / (1024**3 if sys.platform == "darwin" else 1024**2)
print(f"{engine} done in {time.perf_counter() - suite_start:.1f}s, peak memory {peak_gb:.1f} GB", flush=True)
with open(out_path, "w") as fh:
    json.dump({"queries": results, "hits_rows": hits_rows, "peak_gb": peak_gb}, fh)

## 5. Run the benchmark

This cell takes the longest. It prints a line as each query finishes, DuckDB's pass first and then Opteryx's.

**Allocator.** On Linux, the Opteryx performance suites run the engine with the mimalloc build bundled with draken preloaded (`LD_PRELOAD`), so a benchmark measures the allocator production uses. This cell does the same for the Opteryx process. A preload only takes effect when a process starts, which is why the benchmark runs as a script. Any allocator the runtime already preloads is replaced for that process, not stacked. On macOS the suites deliberately use the system allocator, and so does this notebook. DuckDB runs with the runtime's environment unchanged, as it does in the suites' own DuckDB baseline runs.

In [ ]:
import subprocess
import sys

import draken


def run_engine(engine, extra_env):
    out_path = f"results_{engine}.json"
    proc = subprocess.Popen(
        [sys.executable, "bench_clickbench.py", engine, str(ITERATIONS), out_path],
        env={**os.environ, **extra_env},
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"{engine} benchmark exited with code {proc.returncode}")
    with open(out_path) as fh:
        return json.load(fh)


opteryx_env = {}
preload = draken.preload_library_path() if sys.platform.startswith("linux") else None
if preload:
    opteryx_env = {"LD_PRELOAD": preload, "MIMALLOC_PURGE_DELAY": "1000"}
print("Runtime LD_PRELOAD: ", os.environ.get("LD_PRELOAD") or "(none)")
print("Opteryx allocator:  ", f"mimalloc ({preload})" if preload else "system malloc")
print()

by_engine = {
    "duckdb": run_engine("duckdb", {}),
    "opteryx": run_engine("opteryx", opteryx_env),
}

results = []  # rows of dicts, one per query
for name in QUERIES:
    row = {"query": name}
    for engine, engine_results in by_engine.items():
        r = engine_results["queries"][name]
        row[f"{engine}_ms"] = r["ms"]
        row[f"{engine}_median_ms"] = r["median_ms"]
        row[f"{engine}_rows"] = r["rows"]
        row[f"{engine}_error"] = r["error"]
    results.append(row)

## 6. Results

In [ ]:
import pandas as pd

df = pd.DataFrame(results).set_index("query")

hits_rows = {engine: r["hits_rows"] for engine, r in by_engine.items()}
if len(set(hits_rows.values())) > 1:
    print("ENGINES SEE DIFFERENT DATA:", hits_rows)

# Row-count cross-check: the engines should agree, query by query.
row_cols = ["duckdb_rows", "opteryx_rows"]
mismatches = df[row_cols].apply(lambda r: len(set(v for v in r if v is not None)) > 1, axis=1)
if mismatches.any():
    print("ROW COUNT MISMATCH on:", list(df.index[mismatches]))
    display(df.loc[mismatches, row_cols])
else:
    print("Row counts agree across both engines on every query that ran.")

timing_cols = ["duckdb_ms", "opteryx_ms"]
display(df[timing_cols + row_cols].round(1))

errors = df[["duckdb_error", "opteryx_error"]].dropna(how="all")
if not errors.empty:
    print("\nFailures:")
    display(errors)

compared = df[timing_cols].dropna()
print("Total time, queries both engines completed "
      f"({len(compared)}/{len(df)} queries):")
for engine, col in zip(("DuckDB", "Opteryx"), timing_cols):
    print(f"  {engine:<8} {compared[col].sum():>10,.1f} ms")

print("\nPeak memory, each engine's own process:")
for engine, label in (("duckdb", "DuckDB"), ("opteryx", "Opteryx")):
    print(f"  {label:<8} {by_engine[engine]['peak_gb']:>6.1f} GB")

## 7. Charts

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rows_label = f"{NUM_FILES} of 100 partitions"

fig, ax = plt.subplots(figsize=(16, 6))
x = np.arange(len(df))
width = 0.38
colors = {"duckdb_ms": "#f0b429", "opteryx_ms": "#20bf6b"}
labels = {"duckdb_ms": "DuckDB (Parquet)", "opteryx_ms": "Opteryx (Parquet)"}

for i, col in enumerate(timing_cols):
    values = df[col].fillna(0)
    offset = (i - (len(timing_cols) - 1) / 2) * width
    ax.bar(x + offset, values, width, label=labels[col], color=colors[col])

ax.set_yscale("log")
ax.set_xticks(x)
ax.set_xticklabels(df.index, rotation=90)
ax.set_ylabel("min time (ms, log scale)")
ax.set_title(f"ClickBench, {rows_label}: per-query min time across {ITERATIONS} warm runs")
ax.legend()
ax.grid(axis="y", alpha=0.3, which="both")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
engines = ["DuckDB", "Opteryx"]
values = [compared[c].sum() for c in timing_cols]
bar_colors = [colors[c] for c in timing_cols]
ax.bar(engines, values, color=bar_colors)
ax.set_ylabel("total time (ms)")
ax.set_title(f"ClickBench, {rows_label}: total time, {len(compared)} comparable queries")
for i, v in enumerate(values):
    ax.text(i, v, f" {v:,.0f} ms", ha="center", va="bottom")
fig.tight_layout()
plt.show()

## Notes and caveats

- **Not comparable with ClickBench.** This is a subset of the data on a different machine, timed differently. The official results use all 100 partitions on a `c6a.4xlarge`, and each query's first try runs in a fresh process with the OS page cache dropped. Here every iteration is warm and the minimum is reported. Opteryx's official entry is [`opteryx/`](https://github.com/ClickHouse/ClickBench/tree/main/opteryx) in the ClickBench repository, and its results are on [benchmark.clickhouse.com](https://benchmark.clickhouse.com/).
- **Subset effects.** Q37 to Q43 filter on `CounterID = 62` and July 2013, and several use `OFFSET 1000` or `OFFSET 10000`. On a few partitions they can return zero rows, which says little about how either engine handles the full query.
- **One process per engine, one query at a time.** There is no concurrency and no cold-cache control beyond the warm-up. The OS page cache warms naturally, and the Parquet files are shared, so DuckDB's pass also warms the page cache for Opteryx's.
- **Row-count cross-check.** The check in §6 catches query-translation bugs. If the engines disagree on row counts for a query, treat that query's timing as suspect.
- **Scaling up.** Raise `NUM_FILES` in the configuration cell and re-run from §1; only the missing partitions download. The heaviest `GROUP BY` queries need more memory as the data grows. §6 prints each engine's peak; the engines run one after the other, so the larger of the two is what has to fit. A free Colab runtime has about 12.7 GB.
- **Memory includes a cache.** Opteryx keeps decompressed Parquet chunks in a cross-query cache sized from the machine's memory, and the cache gives memory back when a query needs it. On a machine with more memory, Opteryx's peak is higher for the same data. Set the `CHUNK_CACHE_MEMORY_PERCENT` environment variable before running the benchmark to change the cache's share.
- **Not a published result.** As the [Benchmarking](https://docs.opteryx.app/docs/reference/internals/benchmarking#what-the-numbers-are-not) page says, numbers from one machine say nothing about another. This notebook exists so you can measure for yourself.